# Insectes : classification vs détection vs pose — où le modèle regarde-t-il ?

Quatre modèles, une seule tâche : attribuer l'image à l'un des groupes.

| modèle | supervision | dataset | peut s'abstenir ? |
|---|---|---|---|
| `cls` | label global | `AllSpecies-cls` | non |
| `cls_bg` | label global + classe `background` | `AllSpecies-cls-bg` | oui (prédit `background`) |
| `detect` | boîtes | `AllSpecies-detect` | oui (aucune boîte) |
| `pose` | boîtes + keypoints | `AllSpecies-pose` | oui (aucune boîte) |

`cls` et `cls_bg` forment la comparaison **avant / après** l'ajout de la classe `background`,
sur les métriques comme sur l'attention.

À lancer après `fuze_datasets.py` puis `create_background_class.py`.

### Métriques

Une seule chose est mesurée : **la performance par classe**, sous deux formes.

- **Matrice de confusion**, avec une colonne `abstention` supplémentaire. L'abstention n'est
  donc pas une métrique séparée : elle est visible là où elle se produit, classe par classe.
  La diagonale normalisée par ligne donne l'**accuracy par classe** (= rappel).
- **F1 one-vs-rest par classe**, où l'abstention compte comme faux négatif.

### Deux z-scores, deux questions

| | question | dénominateur |
|---|---|---|
| `z_intra` | ce modèle est-il relativement faible sur cette classe ? | écart-type des F1 **entre classes** |
| `z_diff` | l'écart avant/après est-il réel ou du bruit ? | écart-type **bootstrap** de la différence |

`z_intra` est celui décrit littéralement dans une approche one-vs-rest, mais avec 3 ou 4
classes son dénominateur repose sur 3 ou 4 points : il ordonne les classes, il ne teste rien.
`z_diff` est le seul défendable pour comparer deux modèles, parce qu'il estime la variance par
rééchantillonnage des images de test.

## 1. Configuration

Seule cellule à éditer.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path("../datasets")

# nom -> (tache ultralytics, dataset)
MODEL_SPECS = {
    "cls":    ("classify", ROOT / "AllSpecies-cls"),                     # sans background
    "cls_bg": ("classify", ROOT / "AllSpecies-cls-bg"),                  # avec background
    "detect": ("detect",   ROOT / "AllSpecies-detect" / "yolo-config.yaml"),
    "pose":   ("pose",     ROOT / "AllSpecies-pose"   / "yolo-config.yaml"),
}
NAMES = list(MODEL_SPECS)
BEFORE, AFTER = "cls", "cls_bg"          # couple compare avant/apres

CF_DIR      = ROOT / "counterfactual"
CF_VARIANTS = ["mean_noise", "telea", "gray"]
CF_TRAINED  = "mean_noise"               # variante vue par cls_bg a l'entrainement
CF_METRIC   = "telea"                    # variante servant de classe `background` en cellule 5
BACKGROUND  = "background"
BG_BALANCE  = True                       # sous-echantillonne background a la taille d'un groupe

IMGSZ, EPOCHS, BATCH, SCALE, SEED = 640, 100, 16, "n", 0
DEVICE = 0 if torch.cuda.is_available() else "cpu"

SPLIT       = "test"
N_PER_GROUP = 10        # images expliquees par groupe
CONF        = 0.10      # seuil bas : on veut voir l'abstention, pas la masquer
OCC_PATCH, OCC_STRIDE = 96, 48
TARGET_LAYER = None     # None -> dernier bloc C2PSA du backbone
N_BOOTSTRAP  = 2000     # pour z_diff

torch.manual_seed(SEED); np.random.seed(SEED)
RUNS = Path("runs"); RUNS.mkdir(exist_ok=True)

POSE_CFG = yaml.safe_load(open(MODEL_SPECS["pose"][1]))
_n = POSE_CFG["names"]
GROUPS = [_n[i] for i in sorted(_n)] if isinstance(_n, dict) else list(_n)
NC, CHANCE = len(GROUPS), 1.0 / len(GROUPS)
CLASSES = GROUPS + [BACKGROUND]      # espace de sortie commun aux 4 modeles
NCB     = len(CLASSES)
BG_ID   = NC                          # indice de `background` / non-detection
print("device:", DEVICE, "| groupes:", GROUPS)

device: 0 | groupes: ['coleoptera', 'diptera', 'hymenoptera', 'lepidoptera']


## 2. Letterbox, masque, split

Le letterbox est **identique** à celui écrit sur disque par `fuze_datasets.py` pour les
datasets `cls`, et à celui qu'Ultralytics applique en interne pour `detect`/`pose`. C'est cette
identité qui rend les cartes de saillance superposables entre les quatre modèles.

In [2]:
def letterbox(img, size=IMGSZ):
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))
    canvas = np.full((size, size, 3), 114, np.uint8)
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, r, px, py


def load_square(path, size=IMGSZ):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    if img.shape[0] == size and img.shape[1] == size:
        return img                    # deja letterboxee (dataset cls, contre-factuel)
    return letterbox(img, size)[0]


def bbox_mask(label_path, image_path, size=IMGSZ):
    """Masque insecte dans le repere letterboxe."""
    h, w = cv2.imread(str(image_path)).shape[:2]
    r = min(size / h, size / w)
    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))
    px, py = (size - nw) // 2, (size - nh) // 2
    m = np.zeros((size, size), np.uint8)
    for line in Path(label_path).read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cx, cy, bw, bh = map(float, p[1:5])
        x1, y1 = int((cx - bw / 2) * w * r) + px, int((cy - bh / 2) * h * r) + py
        x2, y2 = int((cx + bw / 2) * w * r) + px, int((cy + bh / 2) * h * r) + py
        m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 255
    return m


def norm01(a):
    a = np.nan_to_num(a.astype(np.float32))
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-12 else (a - lo) / (hi - lo)


def load_split(split=SPLIT):
    root  = Path(POSE_CFG["path"])
    imdir = root / POSE_CFG.get(split, f"images/{split}")
    rows = []
    for img in sorted(imdir.rglob("*")):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".webp"}:
            continue
        lbl = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
        if not lbl.exists() or not lbl.read_text().strip():
            continue
        lines = [l for l in lbl.read_text().strip().splitlines() if l.strip()]
        cid = int(float(lines[0].split()[0]))
        rows.append({"image": str(img), "label": str(lbl), "n_instances": len(lines),
                     "class_id": cid, "group": GROUPS[cid]})
    return pd.DataFrame(rows)


test_df = load_split()
print(f"{len(test_df)} images dans le split '{SPLIT}'")
print(test_df.group.value_counts().to_string())

# Hypothese du protocole : UN insecte par image. C'est elle qui autorise a traiter
# une non-detection comme une prediction `background` (cellule 5). Si elle est
# fausse sur certaines images, ces images faussent la classe background.
multi = test_df[test_df.n_instances != 1]
if len(multi):
    print(f"\nATTENTION : {len(multi)} images n'ont pas exactement 1 instance "
          f"({sorted(multi.n_instances.unique())}).")
    print("  L'assimilation non-detection -> background n'est valable que pour 1 insecte.")
    print("  Les retirer :  test_df = test_df[test_df.n_instances == 1]")
else:
    print("\n[ok] exactement 1 insecte par image : non-detection == background.")

255 images dans le split 'test'
group
coleoptera     103
lepidoptera     68
diptera         49
hymenoptera     35

[ok] exactement 1 insecte par image : non-detection == background.


## 3. Entraînement des quatre modèles

Hyperparamètres appariés. `cls` et `cls_bg` ne diffèrent que par leur dataset — c'est ce qui rend leur comparaison interprétable.

In [3]:
BASE = {"classify": f"yolo26{SCALE}-cls.pt", "detect": f"yolo26{SCALE}.pt",
        "pose": f"yolo26{SCALE}-pose.pt"}

def train(name):
    task, data = MODEL_SPECS[name]
    out = RUNS / task / "runs" / "train" / name / "weights" / "best.pt"
    if out.exists():
        print(f"{name}: deja entraine -> {out}")
        return out
    YOLO(BASE[task]).train(data=str(data), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                           seed=SEED, device=DEVICE, project=str(RUNS / "train"),
                           name=name, exist_ok=True, deterministic=True)
    return out

models = {n: YOLO(str(train(n))) for n in NAMES}
for n, m in models.items():
    print(f"{n:<7} classes: {m.names}")

cls: deja entraine -> runs/classify/runs/train/cls/weights/best.pt
New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/AllSpecies-cls-bg, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

## 4. Prédiction image-level unifiée

`detect` / `pose` : classe de la boîte la plus confiante ; aucune boîte = abstention.
`cls_bg` : classe la plus probable ; `background` = abstention.
`cls` : classe la plus probable, jamais d'abstention.

Le remappage passe par `model.names`, pas par l'ordre des dossiers : Ultralytics ordonne les
classes de classification **alphabétiquement**. Avec des groupes en minuscules, `background`
passerait en tête et décalerait tous les indices. Ce remappage rend le notebook insensible au
piège, et il vaut aussi pour `cls` et `cls_bg` qui n'ont pas le même nombre de classes.

In [4]:
MAP = {}
for n in NAMES:
    task = MODEL_SPECS[n][0]
    raw = models[n].names
    raw = raw if isinstance(raw, dict) else dict(enumerate(raw))
    name2idx = {v: k for k, v in raw.items()}
    missing = [g for g in GROUPS if g not in name2idx]
    assert not missing, f"{n} : groupes absents {missing} (vu : {list(name2idx)})"
    MAP[n] = {"task": task,
              "cols": [name2idx[g] for g in GROUPS],
              "bg": name2idx.get(BACKGROUND)}
    print(f"{n:<7} colonnes groupes {MAP[n]['cols']}  background -> {MAP[n]['bg']}")

assert MAP[AFTER]["bg"] is not None, (
    f"{AFTER} n'a pas de classe '{BACKGROUND}'. Lancer create_background_class.py "
    "puis reentrainer.")
assert MAP[BEFORE]["bg"] is None, f"{BEFORE} ne devrait pas avoir de classe '{BACKGROUND}'."


def model_class_index(name, group_idx):
    """Indice de la classe dans l'espace de sortie du modele (Grad-CAM en a besoin)."""
    return MAP[name]["cols"][group_idx]


def predict(name, images, chunk=32):
    """-> scores (N, NC) par groupe, abstain (N,) booleen."""
    info, model = MAP[name], models[name]
    scores, abstain = [], []
    for i in range(0, len(images), chunk):
        kw = dict(imgsz=IMGSZ, verbose=False, device=DEVICE)
        if info["task"] != "classify":
            kw["conf"] = CONF
        for r in model.predict(images[i:i + chunk], **kw):
            if info["task"] == "classify":
                p = r.probs.data.cpu().numpy().astype(np.float32)
                scores.append(p[info["cols"]])
                abstain.append(info["bg"] is not None and int(p.argmax()) == info["bg"])
            else:
                s = np.zeros(NC, np.float32)
                empty = r.boxes is None or len(r.boxes) == 0
                if not empty:
                    c = r.boxes.cls.cpu().numpy().astype(int)
                    f = r.boxes.conf.cpu().numpy()
                    for k in range(NC):
                        if (c == k).any():
                            s[k] = f[c == k].max()
                scores.append(s)
                abstain.append(empty)
    return np.stack(scores), np.array(abstain, bool)


def labels(scores, abstain):
    """-> indice de groupe, ou BG_ID (= NC) si abstention.

    Un seul insecte par image : une non-detection n'est pas une valeur manquante,
    c'est l'affirmation "pas d'insecte ici", donc exactement la classe `background`
    du modele cls_bg. Les quatre modeles partagent alors le meme espace de sortie
    a NC+1 classes, et leurs matrices de confusion sont directement comparables.
    """
    return np.where(abstain, BG_ID, scores.argmax(1))


def n_boxes(name, images, chunk=32):
    """Nombre de boites par image — doit valoir 0 ou 1 sous l'hypothese 1 insecte."""
    if MAP[name]["task"] == "classify":
        return np.zeros(len(images), int)
    out = []
    for i in range(0, len(images), chunk):
        for r in models[name].predict(images[i:i + chunk], imgsz=IMGSZ, conf=CONF,
                                      verbose=False, device=DEVICE):
            out.append(0 if r.boxes is None else len(r.boxes))
    return np.array(out)


imgs   = [load_square(p) for p in test_df.image]
y_true = test_df.class_id.to_numpy()
PRED   = {}
for n in NAMES:
    sc, ab = predict(n, imgs)
    PRED[n] = labels(sc, ab)
    extra = ""
    if MAP[n]["task"] != "classify":
        nb_ = n_boxes(n, imgs)
        if (nb_ > 1).any():
            extra = f"  [{(nb_ > 1).mean():.0%} d'images a boites multiples]"
    print(f"{n:<7} {(PRED[n] == BG_ID).sum():3d} predictions `{BACKGROUND}`{extra}")

cls     colonnes groupes [0, 1, 2, 3]  background -> None
cls_bg  colonnes groupes [1, 2, 3, 4]  background -> 0
detect  colonnes groupes [0, 1, 2, 3]  background -> None
pose    colonnes groupes [0, 1, 2, 3]  background -> None
cls       0 predictions `background`
cls_bg  147 predictions `background`
detect    0 predictions `background`  [4% d'images a boites multiples]
pose      0 predictions `background`  [5% d'images a boites multiples]


## 5. Matrices de confusion et F1 one-vs-rest

**Un seul insecte par image.** Une non-détection n'est donc pas une réponse manquante : c'est
l'affirmation « il n'y a pas d'insecte ici ». Elle est donc reportée sur la classe
`background`, exactement celle que `cls_bg` peut prédire. Les quatre modèles partagent alors
un espace de sortie identique à NC+1 classes, et leurs matrices de confusion se comparent
directement — plus besoin d'une colonne `abstention` à part.

Le jeu d'évaluation combine donc :

- les images du split test, étiquetées par leur groupe ;
- leurs versions **sans insecte** (variante `CF_METRIC`), étiquetées `background`.

`cls`, qui n'a pas de classe `background`, ne peut jamais prédire cette classe : son F1 vaut 0
sur cette ligne et son rappel `background` est nul. Ce n'est pas une anomalie, c'est le coût
exact de l'absence de mécanisme de refus — et c'est la cellule 11 (AUROC sur seuil de
confiance) qui lui rend justice.

La diagonale normalisée par ligne donne l'**accuracy par classe**. Pour le F1 one-vs-rest,
prédire `background` sur un vrai *Coleoptera* est un faux négatif de *Coleoptera* et un faux
positif de `background` — le comptage est le même pour les quatre modèles.

In [5]:
def confusion(y, yp, k=None):
    """(k, k) — lignes = vraie classe, colonnes = predite. `background` inclus."""
    k = k or NCB
    cm = np.zeros((k, k), int)
    for t, pr in zip(y, yp):
        cm[t, pr] += 1
    return cm


def f1_one_vs_rest(y, yp, k=None):
    """F1 par classe, `background` compris."""
    k = k or NCB
    out = np.zeros(k)
    for c in range(k):
        tp = int(((yp == c) & (y == c)).sum())
        fp = int(((yp == c) & (y != c)).sum())
        fn = int(((y == c) & (yp != c)).sum())
        out[c] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    return out


# --- jeu d'evaluation : images reelles + leurs versions sans insecte ---------
idx_cf = {p.stem: p for p in (CF_DIR / CF_METRIC / SPLIT).glob("*")
          if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
has_cf = [i for i, s in enumerate([Path(p).stem for p in test_df.image]) if s in idx_cf]
assert has_cf, f"Aucune image dans {CF_DIR / CF_METRIC / SPLIT}."

bg_pick = has_cf
if BG_BALANCE:
    # sans cela `background` pese 50 % du jeu d'evaluation, ce qui gonfle sa
    # precision et deprime celle des groupes. On le ramene a la taille moyenne
    # d'un groupe pour que les NC+1 classes soient comparables entre elles.
    n_bg = max(1, len(test_df) // NC)
    bg_pick = list(np.random.default_rng(SEED).permutation(has_cf)[:n_bg])

eval_imgs = imgs + [load_square(idx_cf[Path(test_df.image.iloc[i]).stem]) for i in bg_pick]
y_eval    = np.concatenate([y_true, np.full(len(bg_pick), BG_ID)])
print(f"jeu d'evaluation : {len(y_true)} images a insecte + {len(bg_pick)} sans insecte "
      f"(variante '{CF_METRIC}')")
print("effectifs :", {c: int((y_eval == k).sum()) for k, c in enumerate(CLASSES)})

PRED_EVAL = {}
for n in NAMES:
    sc, ab = predict(n, eval_imgs)
    PRED_EVAL[n] = labels(sc, ab)

CM = {n: confusion(y_eval, PRED_EVAL[n]) for n in NAMES}
F1 = {n: f1_one_vs_rest(y_eval, PRED_EVAL[n]) for n in NAMES}

fig, axes = plt.subplots(1, len(NAMES), figsize=(4.2 * len(NAMES), 3.9))
for ax, n in zip(np.atleast_1d(axes), NAMES):
    cmn = CM[n] / np.maximum(CM[n].sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NCB)); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(NCB)); ax.set_yticklabels(CLASSES, fontsize=7)
    for a in range(NCB):
        for b in range(NCB):
            ax.text(b, a, f"{cmn[a, b]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[a, b] > .5 else "black")
    ax.set_title(n, fontsize=10); ax.set_xlabel("predit", fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("vrai", fontsize=8)
plt.tight_layout(); plt.show()

acc = pd.DataFrame({n: np.diag(CM[n]) / np.maximum(CM[n].sum(1), 1) for n in NAMES},
                   index=CLASSES)
print("accuracy par classe (rappel)\n")
print(acc.round(3).to_string())
print(f"\n{BEFORE} ne peut pas predire `{BACKGROUND}` : sa ligne vaut 0 par construction. "
      f"Voir cellule 11.")

jeu d'evaluation : 255 images a insecte + 63 sans insecte (variante 'telea')
effectifs : {'coleoptera': 103, 'diptera': 49, 'hymenoptera': 35, 'lepidoptera': 68, 'background': 63}


<Figure size 1680x390 with 4 Axes>

accuracy par classe (rappel)

              cls  cls_bg  detect   pose
coleoptera   1.00   0.699   1.000  1.000
diptera      0.98   0.000   1.000  1.000
hymenoptera  1.00   0.286   1.000  0.971
lepidoptera  1.00   0.353   1.000  1.000
background   0.00   0.968   0.841  0.937

cls ne peut pas predire `background` : sa ligne vaut 0 par construction. Voir cellule 11.


### z-scores

`z_intra` situe chaque classe **à l'intérieur d'un modèle** : positif = classe mieux reconnue
que la moyenne des classes de ce modèle. Avec peu de classes son écart-type repose sur peu de
points, il ordonne mais ne teste rien.

`z_diff` compare **deux modèles sur la même classe**. La variance vient d'un bootstrap apparié
sur les images de test : on rééchantillonne les mêmes indices pour les deux modèles, ce qui
élimine la variabilité due au choix des images et ne laisse que l'écart entre modèles.
`|z| > 2` ≈ écart difficilement attribuable au hasard d'échantillonnage. Un test réduit rendra
ces valeurs instables — regarder aussi l'intervalle.

La ligne `background` est **structurelle**, pas empirique : `cls` ne peut pas prédire cette
classe, le gain est donc de 0 vers ce que vaut `cls_bg`. Ce n'est pas un résultat. Le résultat
est ailleurs : l'ajout de `background` **dégrade-t-il** la discrimination entre insectes ? Il
faut donc lire les lignes des groupes, et le F1 macro restreint aux groupes, affiché en bas.

In [6]:
def z_intra(f1):
    s = f1.std(ddof=0)
    return (f1 - f1.mean()) / s if s > 1e-12 else np.zeros_like(f1)


def z_diff(y, yp_a, yp_b, n_boot=N_BOOTSTRAP, seed=SEED):
    """Bootstrap apparie : memes indices reechantillonnes pour les deux modeles."""
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.zeros((n_boot, NCB))
    for b in range(n_boot):
        i = rng.integers(0, n, n)
        diffs[b] = f1_one_vs_rest(y[i], yp_b[i]) - f1_one_vs_rest(y[i], yp_a[i])
    obs = f1_one_vs_rest(y, yp_b) - f1_one_vs_rest(y, yp_a)
    sd = diffs.std(0, ddof=1)
    z = np.where(sd > 1e-12, obs / np.maximum(sd, 1e-12), 0.0)
    lo, hi = np.percentile(diffs, [2.5, 97.5], axis=0)
    return obs, z, lo, hi


f1_tab = pd.DataFrame({n: F1[n] for n in NAMES}, index=CLASSES)
zi_tab = pd.DataFrame({n: z_intra(F1[n]) for n in NAMES}, index=CLASSES)
print("F1 one-vs-rest (background inclus)\n"); print(f1_tab.round(3).to_string())
print("\nz_intra (par modele, entre classes)\n"); print(zi_tab.round(2).to_string())

obs, z, lo, hi = z_diff(y_eval, PRED_EVAL[BEFORE], PRED_EVAL[AFTER])
comp = pd.DataFrame({f"F1_{BEFORE}": F1[BEFORE], f"F1_{AFTER}": F1[AFTER],
                     "delta": obs, "z_diff": z, "IC95_bas": lo, "IC95_haut": hi},
                    index=CLASSES)
print(f"\ncomparaison {BEFORE} -> {AFTER} (ajout de la classe {BACKGROUND})\n")
print(comp.round(3).to_string())
for g, zz, d in zip(CLASSES, z, obs):
    tag = "  (attendu : cls ne peut pas predire cette classe)" if g == BACKGROUND else ""
    if abs(zz) > 2:
        print(f"  {g:<14} {'gain' if d > 0 else 'perte'} significatif (z={zz:+.2f}){tag}")
    else:
        print(f"  {g:<14} ecart non distinguable du bruit (z={zz:+.2f}){tag}")

print(f"\nComparaison des SEULS groupes (hors {BACKGROUND}) — c'est la que se voit\n"
      f"le cout eventuel de l'ajout de la classe sur la discrimination entre insectes :")
print(f"  F1 macro groupes : {BEFORE}={F1[BEFORE][:NC].mean():.3f}  "
      f"{AFTER}={F1[AFTER][:NC].mean():.3f}")

F1 one-vs-rest (background inclus)

               cls  cls_bg  detect   pose
coleoptera   0.817   0.804   0.990  0.986
diptera      0.960   0.000   1.000  0.980
hymenoptera  0.972   0.444   1.000  0.986
lepidoptera  0.913   0.522   0.944  1.000
background   0.000   0.450   0.914  0.967

z_intra (par modele, entre classes)

              cls  cls_bg  detect  pose
coleoptera   0.23    1.40    0.60  0.19
diptera      0.61   -1.72    0.87 -0.35
hymenoptera  0.65    0.00    0.87  0.17
lepidoptera  0.49    0.30   -0.73  1.55
background  -1.98    0.02   -1.61 -1.56

comparaison cls -> cls_bg (ajout de la classe background)

             F1_cls  F1_cls_bg  delta  z_diff  IC95_bas  IC95_haut
coleoptera    0.817      0.804 -0.013  -0.318    -0.091      0.070
diptera       0.960      0.000 -0.960 -48.371    -0.991     -0.917
hymenoptera   0.972      0.444 -0.528  -5.427    -0.733     -0.350
lepidoptera   0.913      0.522 -0.391  -5.618    -0.539     -0.267
background    0.000      0.450  0.450  

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
f1_tab.plot(kind="bar", ax=axes[0], rot=30)
axes[0].set_title("F1 one-vs-rest par classe (background inclus)", fontsize=10)
axes[0].set_ylim(0, 1); axes[0].legend(fontsize=7)
axes[0].tick_params(axis="x", labelsize=7)

axes[1].bar(CLASSES, obs, yerr=[obs - lo, hi - obs], capsize=4,
            color=["#4C72B0"] * NC + ["#DD8452"])
axes[1].axhline(0, c="k", lw=1)
axes[1].set_title(f"delta F1 : {AFTER} - {BEFORE}  (IC95 bootstrap)", fontsize=10)
axes[1].tick_params(axis="x", labelrotation=30, labelsize=7)
plt.tight_layout(); plt.show()

<Figure size 1100x380 with 2 Axes>

## 6. Grad-CAM

Deux subtilités que masquerait un appel naïf à `predict()` :

1. `predict()` s'exécute sous `inference_mode` : aucun gradient. On appelle donc le réseau
   directement.
2. Pour récupérer le logit de classe sans dépendre de la valeur de retour de `forward` (qui
   change selon la tâche et la version), on pose un hook sur `head.cv3`.

**Point critique.** Sur une tête end-to-end (YOLO26), la branche `one2one_cv3` opère sur des
features **détachées** : aucun gradient ne peut la traverser jusqu'au backbone. La CAM est
donc calculée sur la branche `one2many` (`cv3`), entraînée conjointement mais pas identique à
celle qui produit les prédictions. À mentionner en méthodologie.

In [11]:

# Branche de classification a utiliser pour la CAM.
# Sur un checkpoint end2end, `cv3` (branche one2many) est supprimee a la
# sauvegarde : seule `one2one_cv3` subsiste. On prend celle qui existe, en
# preferant `one2one_cv3` — c'est elle qui produit les predictions evaluees
# ailleurs dans le notebook, donc la CAM explique bien le meme calcul.
CAM_BRANCH_PREFERENCE = ("one2one_cv3", "cv3")
CAM_BRANCH_USED = {}


def head_class_branch(head):
    for attr in CAM_BRANCH_PREFERENCE:
        b = getattr(head, attr, None)
        if b is not None:
            return b, attr
    raise RuntimeError(
        f"Aucune branche de classification sur {type(head).__name__}. "
        f"Attributs candidats : "
        f"{[a for a in dir(head) if a.startswith('cv') or 'one2one' in a]}")


def get_net(name):
    net = models[name].model.to(DEVICE if DEVICE != "cpu" else "cpu").float().eval()
    for p in net.parameters():
        p.requires_grad_(True)
    return net


def get_layer(net):
    if TARGET_LAYER is not None:
        return net.model[TARGET_LAYER]
    c2psa = [m for m in net.model if type(m).__name__.upper().startswith("C2PSA")]
    assert c2psa, "Aucun bloc C2PSA : preciser TARGET_LAYER."
    return c2psa[-1]          # dernier bloc du backbone = seul point commun aux 4 modeles


def gradcam(name, img, group_idx):
    net, task = get_net(name), MAP[name]["task"]
    head, layer = net.model[-1], get_layer(net)
    cls_idx = model_class_index(name, group_idx)
    store, feats, handles = {}, {}, []

    def keep_act(m, i, o):            # ne rien renvoyer : un hook qui retourne une
        store["a"] = o                # valeur remplacerait la sortie du module
        if o.requires_grad:
            o.register_hook(lambda g: store.__setitem__("g", g.detach()))

    handles.append(layer.register_forward_hook(keep_act))
    if task != "classify":
        # On capture les ENTREES de la tete (sorties du neck, encore rattachees
        # au graphe) plutot que les sorties d'une branche. On recalcule ensuite
        # les logits de classe nous-memes : le gradient remonte alors jusqu'au
        # backbone meme quand la tete detache ses features en interne.
        handles.append(head.register_forward_pre_hook(
            lambda m, args: feats.__setitem__("x", args[0])))

    x = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)[None]
    x = (x.float() / 255).to(next(net.parameters()).device)

    prev = head.training
    if task == "classify":
        head.training = True      # logits bruts au lieu du softmax
    # Pour detect/pose on laisse la tete en eval : forcer `training=True`
    # emprunterait la branche one2many, absente du checkpoint -> TypeError.
    try:
        out = net(x)
        if task == "classify":
            o = out[0] if isinstance(out, (list, tuple)) else out
            score = o.reshape(-1)[cls_idx]
        else:
            branch, attr = head_class_branch(head)
            CAM_BRANCH_USED[name] = attr
            maps = [branch[i](feats["x"][i]) for i in range(len(branch))]
            score = torch.cat([m[0, cls_idx].reshape(-1) for m in maps]).max()
        net.zero_grad(set_to_none=True)
        score.backward()
    finally:
        head.training = prev
        for h in handles:
            h.remove()

    a, g = store["a"][0].detach(), store.get("g")
    assert g is not None, "Aucun gradient : verifier TARGET_LAYER."
    cam = (g[0].mean((1, 2), keepdim=True) * a).sum(0).clamp(min=0).cpu().numpy()
    return norm01(cv2.resize(cam, (IMGSZ, IMGSZ)))

## 7. Vérification : la CAM est-elle class-specific ?

Garde-fou minimal. Si les classes produisent la même carte, la couche cible est trop profonde pour discriminer et les heatmaps ne signifient rien. Tant que cette cellule n'affiche pas `OK` partout, ne pas interpréter la suite.

In [12]:
probe = load_square(test_df.image.iloc[0])
for n in NAMES:
    branche = CAM_BRANCH_USED.get(n, "logits (classify)")
    print(f"{n:<7} branche={branche:<13} ecart inter-classes {ecart:.5f}   [{etat}]")
    
    maps = [gradcam(n, probe, c) for c in range(NC)]
    ecart = np.mean([np.abs(maps[0] - m).mean() for m in maps[1:]]) if NC > 1 else 0.0
    etat = "OK" if maps[0].std() > 1e-6 and ecart > 1e-4 else "PROBLEME"
    print(f"{n:<7} ecart inter-classes {ecart:.5f}   [{etat}]")
    if etat == "PROBLEME":
        print(f"         -> essayer TARGET_LAYER parmi {list(get_net(n).model[-1].f)}")

cls     branche=logits (classify) ecart inter-classes 0.10754   [OK]
cls     ecart inter-classes 0.14683   [OK]
cls_bg  branche=logits (classify) ecart inter-classes 0.14683   [OK]
cls_bg  ecart inter-classes 0.10754   [OK]
detect  branche=logits (classify) ecart inter-classes 0.10754   [OK]
detect  ecart inter-classes 0.12968   [OK]
pose    branche=logits (classify) ecart inter-classes 0.12968   [OK]
pose    ecart inter-classes 0.17373   [OK]


In [13]:
for n in ("detect", "pose"):
    h = models[n].model.model[-1]
    print(n, type(h).__name__,
          "| cv3:", type(getattr(h, "cv3", None)).__name__,
          "| one2one_cv3:", type(getattr(h, "one2one_cv3", None)).__name__,
          "| end2end:", getattr(h, "end2end", None))



detect Detect | cv3: NoneType | one2one_cv3: ModuleList | end2end: True
pose Pose26 | cv3: NoneType | one2one_cv3: ModuleList | end2end: True


## 8. Occlusion

On masque une fenêtre glissante et on mesure la chute du score de la vraie classe. Aucun gradient, aucune hypothèse d'architecture : **strictement la même procédure pour les quatre modèles**. C'est la méthode de référence quand elle contredit Grad-CAM.

In [14]:
def occlusion(name, img, group_idx, patch=OCC_PATCH, stride=OCC_STRIDE):
    base = predict(name, [img])[0][0, group_idx]
    coords, variants = [], []
    for yy in range(0, IMGSZ - patch + 1, stride):
        for xx in range(0, IMGSZ - patch + 1, stride):
            v = img.copy()
            v[yy:yy + patch, xx:xx + patch] = 114     # meme gris que le letterbox
            variants.append(v); coords.append((yy, xx))

    drops = base - predict(name, variants)[0][:, group_idx]
    acc = np.zeros((IMGSZ, IMGSZ), np.float32)
    cnt = np.zeros((IMGSZ, IMGSZ), np.float32)
    for (yy, xx), d in zip(coords, drops):
        acc[yy:yy + patch, xx:xx + patch] += d
        cnt[yy:yy + patch, xx:xx + patch] += 1
    return norm01(np.maximum(acc / np.maximum(cnt, 1), 0))


print(f"{(((IMGSZ - OCC_PATCH)//OCC_STRIDE)+1)**2} inferences par image et par modele")

144 inferences par image et par modele


## 9. Attention : localisation, et effet de la classe `background`

- **EBPG** — fraction de la masse de saillance dans l'insecte. `1 - EBPG` = dépendance au fond.
- **ratio** — EBPG normalisé par l'aire du masque. **C'est la colonne à lire** : un EBPG de
  0.40 sur un insecte couvrant 40 % de l'image, c'est le hasard (ratio = 1).

On explique toujours la **vraie** classe, pas la classe prédite : expliquer la classe prédite
mélangerait erreurs de classification et défauts de localisation.

Le tableau final répond à la question qui motive la classe `background` : est-ce que forcer le
modèle à reconnaître le fond le pousse à regarder davantage l'insecte ?

In [15]:
def ebpg(sal, mask):
    tot = sal.sum()
    return float(sal[mask > 0].sum() / tot) if tot > 0 else 0.0


n_per = min(N_PER_GROUP, int(test_df.group.value_counts().min()))
sample = test_df.groupby("group", group_keys=False).sample(n=n_per, random_state=SEED)
print(f"{len(sample)} images expliquees ({n_per} par groupe)")

rows, saliency = [], {}
for n in NAMES:
    for method, fn in (("gradcam", gradcam), ("occlusion", occlusion)):
        for r in sample.itertuples():
            img  = load_square(r.image)
            mask = bbox_mask(r.label, r.image)
            if mask.sum() == 0:
                continue
            sal = fn(n, img, r.class_id)
            saliency[(n, method, r.image)] = sal
            e, area = ebpg(sal, mask), float((mask > 0).mean())
            rows.append({"model": n, "method": method, "group": r.group, "image": r.image,
                         "ebpg": e, "aire": area, "ratio": e / area if area else np.nan})
        print(f"{n}/{method} termine")

sal_df = pd.DataFrame(rows)
print("\nratio EBPG / aire  (1 = hasard)\n")
print(sal_df.pivot_table(index="model", columns="method", values="ratio").round(2).to_string())

40 images expliquees (10 par groupe)
cls/gradcam termine
cls/occlusion termine
cls_bg/gradcam termine
cls_bg/occlusion termine
detect/gradcam termine
detect/occlusion termine
pose/gradcam termine
pose/occlusion termine

ratio EBPG / aire  (1 = hasard)

method  gradcam  occlusion
model                     
cls        1.78       1.29
cls_bg     2.47       1.97
detect     0.97       4.03
pose       0.93       4.28


In [16]:
# --- effet de la classe background sur l'attention, apparie image par image ---
piv = sal_df.pivot_table(index=["method", "group", "image"], columns="model", values="ratio")
piv = piv.dropna(subset=[BEFORE, AFTER])
piv["delta"] = piv[AFTER] - piv[BEFORE]

rng = np.random.default_rng(SEED)
lines = []
for method in sal_df.method.unique():
    d = piv.xs(method, level="method")["delta"].to_numpy()
    boot = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(N_BOOTSTRAP)])
    lines.append({"method": method, "n": len(d), "delta_moyen": d.mean(),
                  "z": d.mean() / boot.std(ddof=1) if boot.std() > 1e-12 else 0.0,
                  "IC95_bas": np.percentile(boot, 2.5),
                  "IC95_haut": np.percentile(boot, 97.5)})
eff = pd.DataFrame(lines)
print(f"effet de l'ajout de {BACKGROUND} sur le ratio de localisation "
      f"({AFTER} - {BEFORE}, apparie par image)\n")
print(eff.round(3).to_string(index=False))
for r in eff.itertuples():
    verdict = ("regarde DAVANTAGE l'insecte" if r.z > 2 else
               "regarde MOINS l'insecte" if r.z < -2 else
               "aucun effet distinguable du bruit")
    print(f"  {r.method:<10} {verdict} (z={r.z:+.2f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sal_df.pivot_table(index="model", columns="method", values="ratio").plot(
    kind="bar", ax=axes[0], rot=0)
axes[0].axhline(1, ls="--", c="k", lw=1, label="hasard")
axes[0].set_title("ratio EBPG / aire par modele", fontsize=10); axes[0].legend(fontsize=8)
for method in sal_df.method.unique():
    axes[1].hist(piv.xs(method, level="method")["delta"], bins=20, alpha=.55, label=method)
axes[1].axvline(0, c="k", lw=1)
axes[1].set_title(f"delta ratio par image : {AFTER} - {BEFORE}", fontsize=10)
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

effet de l'ajout de background sur le ratio de localisation (cls_bg - cls, apparie par image)

   method  n  delta_moyen     z  IC95_bas  IC95_haut
  gradcam 40        0.688 1.655     0.034      1.611
occlusion 40        0.673 1.797    -0.042      1.432
  gradcam    aucun effet distinguable du bruit (z=+1.66)
  occlusion  aucun effet distinguable du bruit (z=+1.80)


<Figure size 1100x380 with 2 Axes>

## 10. Test contre-factuel

La mesure la plus solide du notebook : elle ne dépend d'aucune hypothèse d'explicabilité.
Trois variantes produites par `create_background_class.py`, à lire **ensemble** :

| variante | ce qu'elle fait | rôle |
|---|---|---|
| `mean_noise` | aplat + bruit + raccord flou | méthode d'entraînement de la classe `background` |
| `telea` | inpainting OpenCV, reconstruit le fond | **jamais vue à l'entraînement** |
| `gray` | aplat gris, aucun raccord | contrôle pur : réaction au trou |

Si un modèle se comporte pareil sur `telea` et sur `gray`, il réagit à l'artefact, pas à
l'absence d'insecte.

In [17]:
def cf_index(variant):
    d = CF_DIR / variant / SPLIT
    return {p.stem: p for p in d.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}


idx   = {v: cf_index(v) for v in CF_VARIANTS}
stems = [Path(p).stem for p in test_df.image]
keep  = [i for i, s in enumerate(stems) if all(s in idx[v] for v in CF_VARIANTS)]
print(f"{len(keep)}/{len(stems)} images appariees sur les {len(CF_VARIANTS)} variantes")
assert keep, f"Aucune image trouvee dans {CF_DIR}. Lancer create_background_class.py."

paired = test_df.iloc[keep].reset_index(drop=True)
y_cf   = paired.class_id.to_numpy()

cf_rows = []
for n in NAMES:
    for v in CF_VARIANTS:
        sc, ab = predict(n, [load_square(idx[v][stems[i]]) for i in keep])
        yp = labels(sc, ab)
        cf_rows.append({"model": n, "variante": v,
                        "encore_correct": float((yp == y_cf).mean()),
                        "abstention": float(ab.mean())})  # ab == prediction `background`
cf_df = pd.DataFrame(cf_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, col, titre in zip(axes, ["encore_correct", "abstention"],
                          ["classe encore correcte sans insecte", "taux d'abstention"]):
    cf_df.pivot(index="model", columns="variante", values=col).plot(kind="bar", ax=ax, rot=0)
    if col == "encore_correct":
        ax.axhline(CHANCE, ls="--", c="k", lw=1, label=f"hasard ({CHANCE:.2f})")
    ax.set_title(titre, fontsize=10); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

p_ok = cf_df.pivot(index="model", columns="variante", values="encore_correct")
print("Lecture :\n")
for n in NAMES:
    real, ctrl = float(p_ok.loc[n, "telea"]), float(p_ok.loc[n, "gray"])
    if abs(real - ctrl) < 0.05:
        v = "identique au controle gris -> reagit au trou, pas a l'absence d'insecte. NON CONCLUANT."
    elif real <= CHANCE + 0.05:
        v = "retombe au niveau du hasard -> pas de dependance au fond detectable."
    elif real < 0.5:
        v = "dependance au fond moderee mais superieure au hasard."
    else:
        v = "dependance au fond FORTE."
    print(f"  {n:<7} telea {real:.3f} / gray {ctrl:.3f} -> {v}")

255/255 images appariees sur les 3 variantes


<Figure size 1200x380 with 2 Axes>

Lecture :

  cls     telea 0.698 / gray 0.776 -> dependance au fond FORTE.
  cls_bg  telea 0.020 / gray 0.000 -> identique au controle gris -> reagit au trou, pas a l'absence d'insecte. NON CONCLUANT.
  detect  telea 0.141 / gray 0.008 -> retombe au niveau du hasard -> pas de dependance au fond detectable.
  pose    telea 0.027 / gray 0.027 -> identique au controle gris -> reagit au trou, pas a l'absence d'insecte. NON CONCLUANT.


## 11. Détecter l'absence d'insecte — les quatre approches sur un axe commun

La cellule précédente teste bien les quatre modèles sur les images sans insecte, mais sa
colonne `abstention` ne les compare pas équitablement : `cls` n'a **aucun mécanisme de refus**,
son taux vaut 0 quoi qu'il arrive. Le comparer ainsi à `cls_bg` revient à mesurer une
différence qu'on a créée par construction.

Chaque modèle possède pourtant un **score de rejet** naturel — d'autant plus élevé qu'il juge
l'insecte absent :

| modèle | score de rejet | mécanisme |
|---|---|---|
| `cls` | `1 − max P(groupe)` | seuil de confiance (implicite) |
| `cls_bg` | `P(background)` | classe dédiée (explicite) |
| `detect` / `pose` | `1 − max conf(boîte)` | seuil de détection |

Ces scores ne sont pas sur la même échelle, ce qui interdit de comparer des taux à seuil fixe.
L'**AUROC** résout exactement ça : elle ne dépend que de l'**ordre** des scores, donc elle
répond à « ce modèle sépare-t-il les images avec insecte de celles sans ? » indépendamment du
seuil et de la calibration. `0.5` = incapable de distinguer, `1.0` = séparation parfaite.

C'est la comparaison qui répond à la vraie question : **la classe `background` apporte-t-elle
quelque chose qu'un simple seuil de confiance sur `cls` n'apporterait pas déjà ?**

In [18]:
def rejection_score(name, images):
    """Score dans [0,1], croissant avec la conviction que l'insecte est absent."""
    sc, ab = predict(name, images)
    if MAP[name]["bg"] is not None:
        return 1.0 - sc.sum(1)          # softmax sur 4 classes -> P(background)
    return 1.0 - sc.max(1)              # 1 - confiance max (cls, detect, pose)


def _rankdata(x):
    order = np.argsort(x, kind="mergesort")
    r = np.empty(len(x), float); r[order] = np.arange(1, len(x) + 1)
    xs = x[order]; i = 0
    while i < len(xs):                  # rangs moyens pour les ex aequo
        j = i
        while j + 1 < len(xs) and xs[j + 1] == xs[i]:
            j += 1
        if j > i:
            r[order[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    return r


def auroc(pos, neg):
    """P(score(sans insecte) > score(avec insecte)) — Mann-Whitney, robuste aux ex aequo."""
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    r = _rankdata(np.concatenate([pos, neg]))
    n1, n0 = len(pos), len(neg)
    return float((r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def auroc_ci(pos, neg, n_boot=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [auroc(rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True))
            for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])


# --- scores sur les memes images, avec puis sans insecte ---------------------
real_imgs = [imgs[i] for i in keep]                     # avec insecte (reference)
REJ = {n: {"reel": rejection_score(n, real_imgs)} for n in NAMES}
for n in NAMES:
    for v in CF_VARIANTS:
        REJ[n][v] = rejection_score(n, [load_square(idx[v][stems[i]]) for i in keep])

rows = []
for n in NAMES:
    neg = REJ[n]["reel"]
    for v in CF_VARIANTS:
        pos = REJ[n][v]
        lo, hi = auroc_ci(pos, neg)
        rows.append({"model": n, "variante": v, "auroc": auroc(pos, neg),
                     "IC95_bas": lo, "IC95_haut": hi,
                     "abstention_reel": float((PRED[n][keep] == BG_ID).mean()),
                     "abstention_sans_insecte": float(cf_df.query(
                         "model == @n and variante == @v")["abstention"].iloc[0])})
det = pd.DataFrame(rows)
print("Capacite a detecter l'absence d'insecte (AUROC du score de rejet)\n")
print(det.round(3).to_string(index=False))

Capacite a detecter l'absence d'insecte (AUROC du score de rejet)

 model   variante  auroc  IC95_bas  IC95_haut  abstention_reel  abstention_sans_insecte
   cls mean_noise  0.727     0.685      0.766            0.000                    0.000
   cls      telea  0.730     0.688      0.767            0.000                    0.000
   cls       gray  0.722     0.679      0.760            0.000                    0.000
cls_bg mean_noise  0.940     0.919      0.959            0.576                    1.000
cls_bg      telea  0.778     0.737      0.815            0.576                    0.980
cls_bg       gray  0.825     0.789      0.857            0.576                    1.000
detect mean_noise  1.000     0.999      1.000            0.000                    0.945
detect      telea  0.996     0.993      0.998            0.000                    0.831
detect       gray  1.000     1.000      1.000            0.000                    0.992
  pose mean_noise  1.000     1.000      1.000        

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- AUROC par modele et variante ---
piv_auc = det.pivot(index="model", columns="variante", values="auroc")
piv_auc.plot(kind="bar", ax=axes[0], rot=0)
axes[0].axhline(0.5, ls="--", c="k", lw=1, label="hasard")
axes[0].set_ylim(0.4, 1.02); axes[0].set_ylabel("AUROC")
axes[0].set_title("Separation avec / sans insecte", fontsize=10); axes[0].legend(fontsize=7)

# --- courbes ROC sur telea (variante jamais vue a l'entrainement) ---
V = "telea"
for n in NAMES:
    pos, neg = REJ[n][V], REJ[n]["reel"]
    thr = np.unique(np.concatenate([pos, neg]))[::-1]
    tpr = [(pos >= t).mean() for t in thr]
    fpr = [(neg >= t).mean() for t in thr]
    axes[1].plot([0] + fpr + [1], [0] + tpr + [1],
                 label=f"{n} (AUC={auroc(pos, neg):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("faux positifs (insecte present juge absent)")
axes[1].set_ylabel("vrais positifs (absence detectee)")
axes[1].set_title(f"ROC — variante '{V}'", fontsize=10); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

# --- la classe background bat-elle un simple seuil de confiance ? -----------
print(f"La classe '{BACKGROUND}' apporte-t-elle plus qu'un seuil sur {BEFORE} ?\n")
for v in CF_VARIANTS:
    a_before = float(det.query("model == @BEFORE and variante == @v")["auroc"].iloc[0])
    a_after  = float(det.query("model == @AFTER  and variante == @v")["auroc"].iloc[0])
    rng = np.random.default_rng(SEED)
    boot = []
    for _ in range(N_BOOTSTRAP):
        i_pos = rng.integers(0, len(keep), len(keep))    # bootstrap APPARIE :
        i_neg = rng.integers(0, len(keep), len(keep))    # memes images pour les 2 modeles
        boot.append(auroc(REJ[AFTER][v][i_pos],  REJ[AFTER]["reel"][i_neg])
                    - auroc(REJ[BEFORE][v][i_pos], REJ[BEFORE]["reel"][i_neg]))
    boot = np.array(boot); sd = boot.std(ddof=1)
    z = (a_after - a_before) / sd if sd > 1e-12 else 0.0
    verdict = ("la classe dediee fait mieux" if z > 2 else
               "le seuil de confiance fait mieux" if z < -2 else
               "aucun avantage distinguable du bruit")
    print(f"  {v:<11} AUROC {BEFORE}={a_before:.3f}  {AFTER}={a_after:.3f}  "
          f"z={z:+.2f}  -> {verdict}")
print(f"\nLire en priorite la ligne 'telea' : c'est la seule variante que {AFTER}\n"
      f"n'a jamais vue a l'entrainement (cf. cellule suivante).")

<Figure size 1200x400 with 2 Axes>

La classe 'background' apporte-t-elle plus qu'un seuil sur cls ?

  mean_noise  AUROC cls=0.727  cls_bg=0.940  z=+8.82  -> la classe dediee fait mieux
  telea       AUROC cls=0.730  cls_bg=0.778  z=+1.89  -> aucun avantage distinguable du bruit
  gray        AUROC cls=0.722  cls_bg=0.825  z=+4.50  -> la classe dediee fait mieux

Lire en priorite la ligne 'telea' : c'est la seule variante que cls_bg
n'a jamais vue a l'entrainement (cf. cellule suivante).


## 12. Diagnostic : `cls_bg` a-t-il appris l'artefact ?

Confond spécifique à la classe `background` : les images `mean_noise` sont dans
**l'entraînement** de `cls_bg`. Il peut donc avoir appris la signature de l'artefact (bruit
uniforme, raccord flou) plutôt que l'absence d'insecte.

Le test tient en une comparaison : si `cls_bg` s'abstient massivement sur `mean_noise` (vue)
mais pas sur `telea` (jamais vue), il reconnaît l'artefact. `cls`, `detect` et `pose` n'ont
jamais vu ces images : ils servent de témoins. Un écart chez eux est du bruit, un écart chez
`cls_bg` seul est le symptôme.

In [20]:
ab = cf_df.pivot(index="model", columns="variante", values="abstention")
print(ab.round(3).to_string(), "\n")

ecarts = {n: float(ab.loc[n, CF_TRAINED] - ab.loc[n, "telea"]) for n in NAMES}
temoins = [ecarts[n] for n in NAMES if n != AFTER]
print(f"ecart d'abstention ({CF_TRAINED} vue - telea jamais vue) :")
for n in NAMES:
    print(f"  {n:<7} {ecarts[n]:+.3f}" + ("   <- modele teste" if n == AFTER else "   (temoin)"))
ref = float(np.mean(temoins))
print(f"\nmoyenne des temoins : {ref:+.3f}")

if ecarts[AFTER] - ref > 0.15:
    print(f"\n=> {AFTER} s'abstient beaucoup plus sur la variante vue a l'entrainement.\n"
          "   Il a appris la SIGNATURE DE L'ARTEFACT, pas l'absence d'insecte.\n"
          "   Son abstention ne prouve donc pas qu'il regarde l'insecte, et la\n"
          "   comparaison avant/apres des cellules 5 et 9 est a lire avec cette reserve.\n"
          "   Correctif : entrainer background sur un MELANGE de methodes\n"
          "   (mean_noise + telea + gray) et garder une 4e methode inedite pour le test.")
else:
    print(f"\n=> pas de sur-abstention specifique a la variante d'entrainement :\n"
          f"   l'abstention de {AFTER} porte bien sur l'absence d'insecte.")

variante   gray  mean_noise  telea
model                             
cls       0.000       0.000  0.000
cls_bg    1.000       1.000  0.980
detect    0.992       0.945  0.831
pose      0.961       0.976  0.949 

ecart d'abstention (mean_noise vue - telea jamais vue) :
  cls     +0.000   (temoin)
  cls_bg  +0.020   <- modele teste
  detect  +0.114   (temoin)
  pose    +0.027   (temoin)

moyenne des temoins : +0.047

=> pas de sur-abstention specifique a la variante d'entrainement :
   l'abstention de cls_bg porte bien sur l'absence d'insecte.


## 13. Galerie

Ligne = modèle, colonne = image. Comparer les lignes `cls` et `cls_bg` donne l'effet visuel de la classe `background` sur l'attention.

In [24]:
def show(method, n=4):
    picks = sample.sample(min(n, len(sample)), random_state=SEED)
    fig, axes = plt.subplots(len(NAMES) + 1, len(picks),
                             figsize=(3 * len(picks), 3 * (len(NAMES) + 1)), squeeze=False)
    for j, r in enumerate(picks.itertuples()):
        img, m = load_square(r.image), bbox_mask(r.label, r.image)
        vis = img.copy()
        x, yb, w, h = cv2.boundingRect((m > 0).astype(np.uint8))
        cv2.rectangle(vis, (x, yb), (x + w, yb + h), (255, 255, 255), 2)
        axes[0][j].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[0][j].set_title(r.group, fontsize=9)
        if j == 0:
            axes[0][j].set_ylabel("image", fontsize=10)
        for i, name in enumerate(NAMES, start=1):
            sal = saliency.get((name, method, r.image))
            if sal is None:
                continue
            heat = cv2.applyColorMap((sal * 255).astype(np.uint8), cv2.COLORMAP_JET)
            axes[i][j].imshow(cv2.cvtColor(cv2.addWeighted(heat, .45, img, .55, 0),
                                           cv2.COLOR_BGR2RGB))
            if j == 0:
                axes[i][j].set_ylabel(name, fontsize=10)
    for row in axes:
        for a in row:
            a.set_xticks([]); a.set_yticks([])
    fig.suptitle(method, fontsize=12); plt.tight_layout(); plt.savefig(method+".png") ; plt.show()


show("gradcam")
show("occlusion")

<Figure size 1200x1500 with 20 Axes>

<Figure size 1200x1500 with 20 Axes>

## Comment lire les résultats

1. **`background` est une classe comme les autres.** Un seul insecte par image, donc une
   non-détection affirme « pas d'insecte » : elle se reporte sur `background`. Les quatre
   modèles partagent le même espace à NC+1 classes. La ligne `background` de `cls` vaut 0
   par construction — ne pas en tirer de conclusion, c'est la cellule 11 qui l'évalue
   équitablement.
2. **`z_diff` avant `delta`.** Un écart de F1 de 0.04 sur 40 images de test n'est pas un
   résultat. L'intervalle bootstrap dit s'il faut y croire.
3. **Sur la comparaison avant/après, lire le F1 macro restreint aux groupes.** Le gain sur
   `background` est structurel ; la vraie question est de savoir si `cls_bg` paie ce gain par
   une moins bonne discrimination entre insectes.
4. **`ratio` avant `ebpg`.** Un EBPG élevé sur des insectes qui remplissent l'image ne veut
   rien dire. `ratio > 1` = concentration sur l'insecte supérieure au hasard.
5. **Grad-CAM vs occlusion.** S'ils convergent, la conclusion tient. S'ils divergent,
   l'occlusion l'emporte : Grad-CAM passe par la branche `one2many` et par les gradients, deux
   sources d'artefact que l'occlusion n'a pas.
6. **Le contre-factuel prime sur les cartes.** Une heatmap dit où le modèle *semble* regarder ;
   le contre-factuel dit s'il *a besoin* de l'insecte.
7. **L'AUROC de la cellule 11 prime sur les taux d'abstention.** Un taux à seuil fixe
   dépend de la calibration de chaque modèle ; l'AUROC n'en dépend pas. Et un modèle qui
   s'abstient sur tout aurait un excellent taux sur les images sans insecte — regarder
   `abstention_reel` en même temps.
8. **La cellule 12 conditionne toute lecture de `cls_bg`.** Si elle signale l'apprentissage de
   l'artefact, ni son abstention ni son gain d'attention ne prouvent quoi que ce soit.

9. **`cls` sans classe `background` n'est pas démuni** : son `1 − max P` est un score de
   rejet parfaitement utilisable. Si son AUROC égale celle de `cls_bg`, la classe dédiée
   n'apporte rien qu'un seuil bien choisi n'apporterait déjà.

### Limites

- Grad-CAM des modèles `detect`/`pose` : la CAM est calculée en réappliquant la branche
  de classification de la tête aux features du neck capturées en amont. Sur un checkpoint
  end2end, cette branche détache ses entrées pendant l'entraînement ; ce détachement est un
  artifice d'optimisation, pas une propriété de la fonction calculée, et le contourner pour
  l'analyse est légitime. La branche effectivement utilisée est affichée en cellule 7.
- Masque = bbox → ratio optimiste en valeur absolue, mais comparable entre modèles.
- `cls` et `cls_bg` n'ont pas le même nombre de classes : leurs F1 one-vs-rest restent
  comparables car calculés sur les mêmes images à insecte et les mêmes classes cibles, mais
  leurs probabilités de sortie ne sont pas calibrées de la même façon.
- Tout le protocole repose sur **un insecte par image**. La cellule 2 le vérifie et
  signale les images qui violent l'hypothèse : sur une image à deux insectes, une
  non-détection ne signifie plus « pas d'insecte ».
- Le bootstrap suppose des images de test indépendantes. Si plusieurs images proviennent d'un
  même spécimen ou d'une même séance de prise de vue, les intervalles sont trop étroits.